# Plan B — Data Preparation: LibriSpeech + MUSAN/DNS Noise

Download and prepare a **much larger** speech enhancement dataset.

| Component | Source | Size |
|-----------|--------|------|
| Clean speech | LibriSpeech `train-clean-360` | ~23 GB, ~1k speakers, ~360h |
| Noise | MUSAN "noise" subset + DNS-Challenge noise | ~2 GB combined |
| Mixed output | 3 SNR levels (0, 5, 10 dB) | ~69 GB wavs |

**Runtime note:** This notebook uses CPU only (no GPU needed).
However the download + mixing + feature extraction takes a while.
The final features are saved directly to Google Drive so you never need
to redo this step.

**Workflow:**
1. Download LibriSpeech & noise datasets
2. Mix clean + noise at multiple SNR levels
3. Extract DAC features (clean + noisy)
4. Extract MOSS features (last-layer + multi-layer)
5. Pack everything and save to Drive

**Prerequisites:**
- Colab Pro (for Drive storage — you'll need ~100+ GB free)
- No GPU runtime required (change to CPU to avoid wasting GPU hours)

---

## 0. Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = "/content/speech-enhancement-project"

# Clone or pull latest code
BRANCH       = "main"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} https://github.com/VictorChen2002/Speech-Enhancement-Project.git {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

os.chdir(PROJECT_DIR)
!pip install -q -r requirements.txt

# Drive output paths
DRIVE_DATA_DIR = "/content/drive/MyDrive/speech_enhancement_data"
DRIVE_ARCHIVES_DIR = "/content/drive/MyDrive/archives_large"
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(DRIVE_ARCHIVES_DIR, exist_ok=True)

print(f"Project: {PROJECT_DIR}")
print(f"Drive data: {DRIVE_DATA_DIR}")
print(f"Drive archives: {DRIVE_ARCHIVES_DIR}")

## 1. Download LibriSpeech train-clean-360

This is a ~23 GB download. Files are FLAC format, 16 kHz, mono.
We download to local Colab storage (fast SSD), then extract features.

In [ ]:
import os
os.chdir(PROJECT_DIR)

DATA_RAW = "data_large/raw"
CLEAN_DIR = f"{DATA_RAW}/clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

LIBRI_URL = "https://www.openslr.org/resources/12/train-clean-360.tar.gz"
LIBRI_TAR = "/content/train-clean-360.tar.gz"

if not os.path.exists(LIBRI_TAR):
    print("Downloading LibriSpeech train-clean-360 (~23 GB)...")
    !wget -q --show-progress -O {LIBRI_TAR} {LIBRI_URL}
else:
    print(f"Already downloaded: {LIBRI_TAR}")

# Extract directly into clean dir
print("Extracting...")
!tar xzf {LIBRI_TAR} -C {DATA_RAW}/
# LibriSpeech extracts to: DATA_RAW/LibriSpeech/train-clean-360/
!mv {DATA_RAW}/LibriSpeech/train-clean-360 {CLEAN_DIR}/librispeech
!rm -rf {DATA_RAW}/LibriSpeech

# Count files
n_files = !find {CLEAN_DIR} -name "*.flac" | wc -l
print(f"Clean speech files: {n_files[0].strip()}")

## 2. Download Noise Datasets

**MUSAN** noise subset (~930 files) + **DNS-Challenge** noise (~150 files).
These provide diverse noise conditions (babble, music, ambient, etc.).

In [ ]:
import os
os.chdir(PROJECT_DIR)

NOISE_DIR = "data_large/raw/noise"
os.makedirs(NOISE_DIR, exist_ok=True)

# --- MUSAN noise subset ---
MUSAN_URL = "https://www.openslr.org/resources/17/musan.tar.gz"
MUSAN_TAR = "/content/musan.tar.gz"

if not os.path.exists(MUSAN_TAR):
    print("Downloading MUSAN (~1.4 GB)...")
    !wget -q --show-progress -O {MUSAN_TAR} {MUSAN_URL}
else:
    print(f"Already downloaded: {MUSAN_TAR}")

print("Extracting MUSAN...")
!tar xzf {MUSAN_TAR} -C /content/
# We only want the noise and music subsets (not speech)
!cp -r /content/musan/noise/* {NOISE_DIR}/
!cp -r /content/musan/music/* {NOISE_DIR}/
!rm -rf /content/musan

n_noise = !find {NOISE_DIR} -name "*.wav" | wc -l
print(f"Noise files: {n_noise[0].strip()}")

## 3. Mix Clean + Noise at Multiple SNR Levels

Creates noisy speech at SNR = 0, 5, 10 dB using the existing mixer.

**Tip:** To speed things up, you can use a subset. Set `MAX_CLEAN` below
to limit the number of clean files processed (e.g. 5000 for quick tests).

In [ ]:
import os
os.chdir(PROJECT_DIR)

# ---- Configuration ----
MAX_CLEAN = 0          # 0 = use all files; set e.g. 5000 for quick test
SNR_LEVELS = "0 5 10"  # space-separated
SR = 16000
SEED = 42

CLEAN_DIR = "data_large/raw/clean"
NOISE_DIR = "data_large/raw/noise"
MIXED_DIR = "data_large/mixed"

# If limiting, create a symlinked subset
if MAX_CLEAN > 0:
    import glob, random
    random.seed(SEED)
    all_clean = sorted(glob.glob(f"{CLEAN_DIR}/**/*.flac", recursive=True))
    subset = random.sample(all_clean, min(MAX_CLEAN, len(all_clean)))
    subset_dir = "data_large/raw/clean_subset"
    os.makedirs(subset_dir, exist_ok=True)
    for f in subset:
        dst = os.path.join(subset_dir, os.path.basename(f))
        if not os.path.exists(dst):
            os.symlink(os.path.abspath(f), dst)
    CLEAN_DIR = subset_dir
    print(f"Using subset of {len(subset)} clean files")

cmd = f"python -m src.data.mixer --clean_dir {CLEAN_DIR} --noise_dir {NOISE_DIR} --out_dir {MIXED_DIR} --snr_list {SNR_LEVELS} --sr {SR} --seed {SEED}"
print(f"Command:\n  {cmd}\n")
!{cmd}

# Count output files
for snr in SNR_LEVELS.split():
    d = f"{MIXED_DIR}/snr_{snr}dB"
    n = !find {d} -name "*.wav" 2>/dev/null | wc -l
    print(f"  snr_{snr}dB: {n[0].strip()} files")

## 4. Extract DAC Features (Clean + Noisy)

This runs DAC encoding on **CPU** (slow but works without GPU).
If you have a GPU runtime, it will be used automatically.

We extract features for one SNR level (5 dB is the middle ground).
Feel free to change `SNR_TO_USE` to run others.

In [ ]:
import os
os.chdir(PROJECT_DIR)

SNR_TO_USE = 5  # Which SNR level to use as the "noisy" condition

FEATURES_DIR = "data_large/features"
os.makedirs(FEATURES_DIR, exist_ok=True)

# Extract noisy DAC features
noisy_input = f"data_large/mixed/snr_{SNR_TO_USE}dB"
noisy_output = f"{FEATURES_DIR}/noisy_dac"

if os.path.exists(noisy_output) and len(os.listdir(noisy_output)) > 0:
    print(f"Noisy DAC features already exist ({len(os.listdir(noisy_output))} files), skipping.")
else:
    print("Extracting noisy DAC features...")
    !python -m src.data.extract_dac --audio_dir {noisy_input} --out_dir {noisy_output} --sr 16000

# Extract clean DAC features
# Clean files are .flac in nested dirs, extract_dac handles rglob
clean_input = "data_large/raw/clean"
clean_output = f"{FEATURES_DIR}/clean_dac"

if os.path.exists(clean_output) and len(os.listdir(clean_output)) > 0:
    print(f"Clean DAC features already exist ({len(os.listdir(clean_output))} files), skipping.")
else:
    print("Extracting clean DAC features...")
    !python -m src.data.extract_dac --audio_dir {clean_input} --out_dir {clean_output} --sr 16000

# Verify counts match
n_noisy = len(os.listdir(noisy_output)) if os.path.exists(noisy_output) else 0
n_clean = len(os.listdir(clean_output)) if os.path.exists(clean_output) else 0
print(f"\nDAC features: noisy={n_noisy}, clean={n_clean}")
if n_noisy != n_clean:
    print("⚠️  Count mismatch! Check that mixer output stems match clean stems.")

## 5. Extract MOSS Features — Last Layer

Extracts the final encoder output of MOSS-Audio-Tokenizer (768-dim, 12.5 Hz).
This is used for `last_layer` conditioning.

**Note:** MOSS extraction is slower than DAC. On CPU this can take many hours
for the full dataset. Consider using a GPU runtime for this step, or limiting
to a subset.

In [ ]:
import os
os.chdir(PROJECT_DIR)

SNR_TO_USE = 5
FEATURES_DIR = "data_large/features"
noisy_input = f"data_large/mixed/snr_{SNR_TO_USE}dB"
moss_last_output = f"{FEATURES_DIR}/moss_last"

if os.path.exists(moss_last_output) and len(os.listdir(moss_last_output)) > 0:
    print(f"MOSS last-layer features already exist ({len(os.listdir(moss_last_output))} files), skipping.")
else:
    print("Extracting MOSS last-layer features...")
    !python -m src.data.extract_moss --audio_dir {noisy_input} --out_dir {moss_last_output}

n = len(os.listdir(moss_last_output)) if os.path.exists(moss_last_output) else 0
print(f"\nMOSS last-layer features: {n} files")

## 6. Extract MOSS Features — Multi-Layer (Optional)

Extracts all 32 hidden layers from MOSS Stage-3 transformer (1280-dim each).
This is used for `multi_layer` conditioning.

**Warning:** This generates ~14x more data per file than last-layer.
For the full LibriSpeech-360, this can be ~500+ GB.
Consider skipping this for PoC and only running `last_layer` experiments
on the large dataset.

In [ ]:
import os
os.chdir(PROJECT_DIR)

SNR_TO_USE = 5
FEATURES_DIR = "data_large/features"
noisy_input = f"data_large/mixed/snr_{SNR_TO_USE}dB"
moss_multi_output = f"{FEATURES_DIR}/moss_multi"

# Set EXTRACT_MULTI = True to proceed (disabled by default to save time/space)
EXTRACT_MULTI = False

if EXTRACT_MULTI:
    if os.path.exists(moss_multi_output) and len(os.listdir(moss_multi_output)) > 0:
        print(f"MOSS multi-layer features already exist ({len(os.listdir(moss_multi_output))} files), skipping.")
    else:
        print("Extracting MOSS multi-layer features (this will take a LONG time)...")
        !python -m src.data.extract_moss --audio_dir {noisy_input} --out_dir {moss_multi_output} --save_all_layers

    n = len(os.listdir(moss_multi_output)) if os.path.exists(moss_multi_output) else 0
    print(f"\nMOSS multi-layer features: {n} files")
else:
    print("Skipped multi-layer extraction (set EXTRACT_MULTI = True to enable)")
    print("You can still train with 'none' and 'last_layer' conditioning.")

## 7. Create Train/Validation/Test Split

80/10/10 split, saved to `data_large/split.json`.
This ensures consistent splits across experiments.

In [ ]:
import os, json, random
os.chdir(PROJECT_DIR)

FEATURES_DIR = "data_large/features"
SPLIT_FILE = "data_large/split.json"

# Discover all stems from clean_dac
clean_dac_dir = os.path.join(FEATURES_DIR, "clean_dac")
stems = sorted([f.replace(".pt", "") for f in os.listdir(clean_dac_dir) if f.endswith(".pt")])
print(f"Total stems: {len(stems)}")

random.seed(42)
random.shuffle(stems)

n = len(stems)
n_train = int(0.8 * n)
n_valid = int(0.1 * n)

split = {
    "train": stems[:n_train],
    "valid": stems[n_train:n_train + n_valid],
    "test":  stems[n_train + n_valid:],
}

print(f"Train: {len(split['train'])}, Valid: {len(split['valid'])}, Test: {len(split['test'])}")

with open(SPLIT_FILE, "w") as f:
    json.dump(split, f, indent=2)

print(f"Saved split to {SPLIT_FILE}")

## 8. Pack Features and Save to Drive

Archives the extracted features into tar.gz files on Google Drive.
This way you can reuse them without re-extraction.

In [ ]:
import os
os.chdir(PROJECT_DIR)

FEATURES_DIR = "data_large/features"
DRIVE_ARCHIVES = "/content/drive/MyDrive/archives_large"
os.makedirs(DRIVE_ARCHIVES, exist_ok=True)

import shutil

# Copy split file to Drive
shutil.copy2("data_large/split.json", os.path.join(DRIVE_ARCHIVES, "split_large.json"))

for feat_name in ["clean_dac", "noisy_dac", "moss_last"]:
    feat_path = os.path.join(FEATURES_DIR, feat_name)
    if not os.path.exists(feat_path):
        print(f"Skipping {feat_name} (not found)")
        continue

    archive = os.path.join(DRIVE_ARCHIVES, f"features_{feat_name}.tar.gz")
    if os.path.exists(archive):
        print(f"Archive already exists: {archive}")
        continue

    n = len([f for f in os.listdir(feat_path) if f.endswith('.pt')])
    print(f"Packing {feat_name} ({n} files)...")
    !cd {FEATURES_DIR} && tar czf {archive} {feat_name}/
    size_mb = os.path.getsize(archive) / 1024**2
    print(f"  -> {archive} ({size_mb:.0f} MB)")

# Moss multi (if extracted)
moss_multi_path = os.path.join(FEATURES_DIR, "moss_multi")
if os.path.exists(moss_multi_path) and len(os.listdir(moss_multi_path)) > 0:
    print("\nPacking moss_multi in shards...")
    import glob
    files = sorted(glob.glob(os.path.join(moss_multi_path, "*.pt")))
    SHARD_SIZE = 5000
    for i in range(0, len(files), SHARD_SIZE):
        shard_files = files[i:i+SHARD_SIZE]
        shard_idx = i // SHARD_SIZE
        archive = os.path.join(DRIVE_ARCHIVES, f"features_moss_multi_shard{shard_idx}.tar.gz")
        if os.path.exists(archive):
            print(f"Shard {shard_idx} already exists, skipping")
            continue
        # Create a file list
        list_file = f"/tmp/shard_{shard_idx}.txt"
        with open(list_file, "w") as f:
            for fp in shard_files:
                f.write(f"moss_multi/{os.path.basename(fp)}\n")
        !cd {FEATURES_DIR} && tar czf {archive} -T {list_file}
        size_mb = os.path.getsize(archive) / 1024**2
        print(f"  Shard {shard_idx}: {len(shard_files)} files ({size_mb:.0f} MB)")

print("\n✅ All features saved to Drive!")
print(f"Archives: {DRIVE_ARCHIVES}")
!ls -lh {DRIVE_ARCHIVES}/

## 9. Summary & Next Steps

Your large-scale features are now on Google Drive.

**Next:** Open `notebooks/plan_b_train.ipynb` to train with the larger dataset.

The training notebook will:
1. Unpack features from Drive archives
2. Use the large split file
3. Run epoch-based training with early stopping
4. Evaluate and compare with the original small-data results

In [ ]:
print("Data preparation complete!")
print()
print("Archives on Drive:")
!ls -lh {DRIVE_ARCHIVES}/
print()
print("Next steps:")
print("  1. Switch to a GPU runtime")
print("  2. Open notebooks/plan_b_train.ipynb")
print("  3. Train with the larger dataset")